In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import matplotlib.pyplot as plt

plt.style.use("bmh")
plt.rcParams["axes.axisbelow"] = True
import numpy as np
import pandas
import pandas as pd
from pandas import DataFrame, Index, Series, Timedelta, Timestamp

from tsdm.util.strings import snake2camel

rng = np.random.default_rng()
np.set_printoptions()

## Helper Functions

In [3]:
def data_overview(df: DataFrame):
    overview = DataFrame(index=df.columns)
    mask = pandas.isna(df)
    overview["# datapoints"] = (~mask).sum()
    overview["% missing"] = (mask.mean() * 100).round(2)
    overview["min"] = df.min().round(2)
    overview["mean"] = df.mean().round(2)
    overview["std"] = df.std().round(2)
    overview["max"] = df.max().round(2)
    return overview

## Overview Task data

pretty much the same as cleaned but without run 355

In [4]:
from tsdm.datasets import KIWI_RUNS

ds = KIWI_RUNS()

data = ds.timeseries.copy()
units = ds.units

for run_exp in data.reset_index(level=2).index.unique():
    time = data.loc[run_exp].index
    td = (time.max() - time.min()) / Timedelta("1h")
    data.loc[run_exp, "runtime"] = td

overview = data_overview(data.reset_index(level=[0, 1], drop=True))
overview["unit"] = units.loc[ds.timeseries.columns]

with pd.option_context("display.float_format", "{:,.2f}".format):
    display(overview)

--2022-01-17 16:59:15--  https://owncloud.innocampus.tu-berlin.de/index.php/s/fRBSr82NxY7ratK/download/kiwi_experiments_and_run_355.pk
Resolving owncloud.innocampus.tu-berlin.de (owncloud.innocampus.tu-berlin.de)... 130.149.133.205
Connecting to owncloud.innocampus.tu-berlin.de (owncloud.innocampus.tu-berlin.de)|130.149.133.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 175592441 (167M) [application/octet-stream]
Saving to: ‘/home/rscholz/.tsdm/rawdata/KIWI_RUNS/kiwi_experiments_and_run_355.pk’

     0K .......... .......... .......... .......... ..........  0% 1,45M 1m55s
    50K .......... .......... .......... .......... ..........  0% 8,80M 67s
   100K .......... .......... .......... .......... ..........  0% 2,65M 66s
   150K .......... .......... .......... .......... ..........  0%  366M 49s
   200K .......... .......... .......... .......... ..........  0%  510M 40s
   250K .......... .......... .......... .......... ..........  0% 2,85M 43s
   3

,# datapoints,% missing,min,mean,std,max,unit
variable,,,,,,,
Flow_Air,440912,0.00,0.00,5.52,1.69,10.00,Ln/min
StirringSpeed,440912,0.00,0.00,694.39,956.74,"2,600.00",U/min
Temperature,156960,64.40,32.69,37.30,0.52,41.38,°C
Acetate,1937,99.56,0.00,0.14,0.12,1.18,g/L
Base,17264,96.08,0.00,"1,172.00","1,125.71","4,070.00",uL
Cumulated_feed_volume_glucose,440912,0.00,0.00,398.67,544.06,"4,101.00",uL
Cumulated_feed_volume_medium,440912,0.00,0.00,"1,054.07","1,045.39","5,060.85",uL
DOT,191879,56.48,0.00,71.10,29.53,100.00,%
Glucose,1871,99.58,0.00,1.32,1.71,6.93,g/L


In [5]:
fig, axes = plt.subplots(ncols=4, nrows=4, figsize=(12, 12))

for col, ax in zip(data, axes.flatten()):
    vals = data[col]
    mask = pandas.notna(vals)
    ax.hist(vals[mask], bins=59, density=True)
    ax.set_title(snake2camel(col))
    ax.set_xscale("symlog")
    # ax.set_yscale("log")

## Plotting specific single Experiment

In [6]:
def make_all_plots(key, ts):
    ts = ts.astype("float32")
    T = ((ts.index - ts.index[0]) / Timedelta("1h")).values
    fig, axes = plt.subplots(
        nrows=5, ncols=3, figsize=(10, 14), constrained_layout=True, sharex=True
    )
    for col, ax in zip(ts.columns, axes.flatten()):
        vals = ts[col]
        mask = pandas.notna(vals)
        ax.plot(
            T[mask],
            vals[mask],
            ls="-",
            lw=.5,
            marker=".",
            ms=3,
        )
        ax.set_title(snake2camel(col))

        ymin, ymax = overview["min"][col], overview["max"][col]
        ypad = (ymax - ymin) / 20
        ax.set_ylim(ymin - ypad, ymax + ypad)
        xmin, xmax = 0, overview["max"]["runtime"]
        xpad = (xmax - xmin) / 20
        ax.set_xlim(xmin - xpad, xmax + xpad)
    fig.suptitle(f"Run {key[0]} -- Experiment {key[1]}")
    return fig

In [7]:
ts =  ds.timeseries.copy()
ts = ts[sorted(ts.columns, key=snake2camel)]
key = 439, 15325
ts = ts.loc[key]

fig = make_all_plots(key, ts);

# KIWI_RUNS - The booklet

In [8]:
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm

In [9]:
%matplotlib agg

ts = ds.timeseries
ts = ts[sorted(ts.columns, key=snake2camel)]

with PdfPages("pics/kiwi-runs-booklet.pdf") as pdf:
    groups = ts.groupby(["run_id", "experiment_id"])

    for key, slc in tqdm(groups):
        slc = slc.reset_index(["run_id", "experiment_id"], drop=True)
        fig = make_all_plots(key, slc)
        pdf.savefig(fig)
        plt.close(fig)

  0%|          | 0/264 [00:00<?, ?it/s]